# 02 · 1D Convolutional Neural Network (CNN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LekshmiSR/proteins-d1-hands-on/blob/main/notebooks/02_cnn_1d.ipynb)

**Idea.** A 1D convolution slides a small **filter** (say 9 residues wide) along the protein. At every position it computes a weighted sum of the 9 residues under it. The *same* filter weights are used at every position, so a pattern learned once (e.g. "alternating hydrophobic / polar residues", typical of strands) is detected anywhere in the chain.

```
residues  :  K  T  L  N  E  G  L  G  L  L  E  R  V
filter    :     [w1 w2 w3 w4 w5 w6 w7 w8 w9]           -> one number at the centre position
slides →  :        [w1 w2 w3 w4 w5 w6 w7 w8 w9]        -> next number ...
```

A conv layer has many filters (here 128), each learning a different pattern, so every residue gets a 128-number description. **Stacking** layers lets the network combine small patterns into bigger ones and see further along the chain.

Differences from the MLP of notebook 01:
* **weight sharing**: far fewer weights per position, the same pattern detector everywhere;
* **works on the whole protein at once**: no need to cut windows by hand;
* the context a residue sees (the **receptive field**) grows with every layer.

Expected time on a Colab T4: about 3–5 minutes.

## 0. Setup

Run the next cell first. It works in two places:

* **Google Colab**: it downloads this repository (code + helpers) into the Colab machine.
* **Your own computer** (Jupyter / VS Code): it just moves to the repository folder.

> **Colab tip:** use a GPU. Menu **Runtime → Change runtime type → T4 GPU → Save**. Do this *before* running cells.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/LekshmiSR/proteins-d1-hands-on.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if "google.colab" in sys.modules:                       # running in Colab
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.exists(REPO_DIR):
            subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":      # running locally from notebooks/
    os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from protein_dl import data, train, plots

train.set_seed(42)
device = train.get_device()

## 1. Settings

In [ ]:
EMB_DIM = 64         # size of the learned vector for each amino acid
CHANNELS = 128       # number of filters in every conv layer
KERNEL = 9           # filter width (odd, so it has a centre)
N_LAYERS = 3         # number of stacked conv layers
DROPOUT = 0.2

EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3
MAX_LEN = 512
TRAIN_SUBSET = None  # e.g. 2000 for a quick test
SAVE_TO_DRIVE = False

## 2. Load the data

The same loaders as in notebook 00 (see there for every detail). We grab one batch to push through the model step by step.

In [ ]:
train_dl, valid_dl, test_dl = data.make_dataloaders("data", batch_size=BATCH_SIZE, max_len=MAX_LEN,
                                                     train_subset=TRAIN_SUBSET)
tokens, labels, lengths = next(iter(train_dl))
print("tokens :", tokens.shape, "  <- [B, L]")
print("labels :", labels.shape)
print("lengths:", lengths.shape)

## 3. Build the model step by step

### Step 1: embedding, then "channels first"

PyTorch's `Conv1d` expects tensors shaped **`[B, C, L]`**: batch, channels, length. Our embedding gives `[B, L, E]`, so we swap the last two dimensions with `transpose`. Think of the 64 embedding numbers as 64 "channels" running along the protein, like the red/green/blue channels of an image.

In [ ]:
embedding = nn.Embedding(data.VOCAB_SIZE, EMB_DIM, padding_idx=data.PAD_IDX)
x = embedding(tokens)
print("embedded       :", tuple(x.shape), "  [B, L, E]")
x = x.transpose(1, 2)
print("channels first :", tuple(x.shape), "  [B, E, L]")

### Step 2: one convolution layer

`Conv1d(in_channels=64, out_channels=128, kernel_size=9, padding=4)`

* Each of the 128 filters has shape `[64, 9]`: it looks at all 64 input channels over 9 neighbouring residues.
* `padding=4` adds 4 zeros at each end so the output has the **same length L** as the input (one output per residue).

In [ ]:
conv1 = nn.Conv1d(EMB_DIM, CHANNELS, kernel_size=KERNEL, padding=KERNEL // 2)
print("filter weights:", tuple(conv1.weight.shape), "  [out_channels, in_channels, kernel]")
h = conv1(x)
print("after conv1   :", tuple(h.shape), "  [B, CHANNELS, L]  (same L!)")
h = F.relu(h)
print("after ReLU    :", tuple(h.shape))

Let us verify by hand what the convolution computes at one position: the sum over channels and over the 9 neighbours of `weight × input`, plus a bias.

In [ ]:
b, f, pos = 0, 0, 20                     # protein 0, filter 0, residue 20
patch = x[b, :, pos - 4: pos + 5]        # the 9 residues centred on residue 20: [64, 9]
manual = (conv1.weight[f] * patch).sum() + conv1.bias[f]
print("patch shape        :", tuple(patch.shape))
print(f"manual computation : {manual.item():.5f}")
print(f"Conv1d output      : {conv1(x)[b, f, pos].item():.5f}")

### Step 3: stack layers and watch the receptive field grow

With kernel 9, one layer sees 9 residues. Each extra layer adds 8 more (4 on each side):

receptive field = 1 + N_LAYERS × (KERNEL − 1)

In [ ]:
for n in range(1, N_LAYERS + 1):
    print(f"{n} layer(s): each output sees {1 + n * (KERNEL - 1)} residues")

### Step 4: from 128 channels to 3 classes

A convolution with `kernel_size=1` is just a `Linear` layer applied at every position: it maps the 128 features of each residue to 3 logits. Finally we transpose back to `[B, L, 3]`, the format our loss expects.

In [ ]:
head = nn.Conv1d(CHANNELS, data.NUM_CLASSES, kernel_size=1)
out = head(h)
print("after 1x1 conv :", tuple(out.shape), "  [B, 3, L]")
print("transposed     :", tuple(out.transpose(1, 2).shape), "  [B, L, 3]")

### The model class

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, channels=CHANNELS, kernel=KERNEL, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.embedding = nn.Embedding(data.VOCAB_SIZE, emb_dim, padding_idx=data.PAD_IDX)
        layers, in_ch = [], emb_dim
        for _ in range(n_layers):
            layers += [nn.Conv1d(in_ch, channels, kernel, padding=kernel // 2), nn.ReLU(), nn.Dropout(dropout)]
            in_ch = channels
        self.convs = nn.Sequential(*layers)
        self.head = nn.Conv1d(channels, data.NUM_CLASSES, kernel_size=1)

    def forward(self, tokens, lengths=None):
        x = self.embedding(tokens).transpose(1, 2)   # [B, E, L]
        x = self.convs(x)                            # [B, C, L]
        return self.head(x).transpose(1, 2)          # [B, L, 3]


model = CNN1D().to(device)
print(model)
print(f"\nTrainable parameters: {train.count_parameters(model):,}")
with torch.no_grad():
    print("output shape on our batch:", tuple(model(tokens.to(device), lengths).shape))

> **A note on padding.** At the end of a short protein, the filters also "see" padding positions. Because the padding embedding is fixed at zero, these act like the zero-padding at the protein ends, which is harmless. The loss ignores the padded positions anyway.

## Train

`train.train_model` runs the loop from notebook 01: for each epoch it trains on all training batches, then measures loss and Q3 on the validation set, and remembers the weights of the best epoch. 

In [ ]:
train.set_seed(42)
history = train.train_model(model, train_dl, valid_dl, device, epochs=EPOCHS, lr=LR)

In [ ]:
plots.plot_history(history, "1D CNN")

**Reading the curves:** training loss should go down steadily. If validation loss starts going *up* while training loss keeps going down, the model is **overfitting** (memorising the training proteins). The best-epoch weights protect us from that.

## Evaluate on the test set (CB513)

The model now has the weights from its best validation epoch. We run it **once** on CB513, the independent test set.

In [ ]:
val_metrics = train.evaluate(model, valid_dl, device)
test_metrics = train.evaluate(model, test_dl, device)

print(f"Validation Q3: {val_metrics['q3_accuracy']:.4f}")
print(f"CB513 Q3     : {test_metrics['q3_accuracy']:.4f}   macro F1: {test_metrics['macro_f1']:.4f}")
print()
print("Per-class scores on CB513:")
for cls, s in test_metrics["per_class"].items():
    print(f"  {cls:6s} precision {s['precision']:.3f}  recall {s['recall']:.3f}  F1 {s['f1']:.3f}  ({s['support']} residues)")

In [ ]:
plots.plot_confusion(test_metrics["confusion_matrix"], "1D CNN on CB513 (rows sum to 1)")

**How to read the confusion matrix:** each row is a true class and shows where its residues ended up. The diagonal is the recall of each class. Off-diagonal cells show the typical mistakes (strands predicted as coil is usually the biggest one).

Let us look at a real prediction for one CB513 protein:

In [ ]:
trues, preds, _ = train.predict(model, test_dl, device)
i = 100                                           # try other numbers!
x = np.array(test_dl.dataset.tokens[i]); y = np.array(test_dl.dataset.labels[i])
seq = "".join(data.AMINO_ACIDS[t - 1] if 1 <= t <= 20 else "X" for t in x[y != data.IGNORE_INDEX])
print(f"protein {i}: {len(seq)} scored residues, accuracy {(trues[i] == preds[i]).mean():.2f}\n")
plots.show_prediction(seq, trues[i], preds[i])

## Save the results

The results (metrics, training curves, settings) are written to `results/cnn.json`. Notebook 05 reads these files to compare all models.

In Colab, files disappear when the session ends. Set `SAVE_TO_DRIVE = True` in the settings cell at the top to also keep a copy in your Google Drive (Colab will ask for permission).

In [ ]:
drive_dir = train.get_drive_dir(SAVE_TO_DRIVE)
config = dict(EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR, MAX_LEN=MAX_LEN, TRAIN_SUBSET=TRAIN_SUBSET)
results = train.save_results("cnn", model, history, val_metrics, test_metrics, config=config, drive_dir=drive_dir)

## What did we learn?

* With weight sharing, the CNN usually matches or beats the MLP while reusing the same pattern detectors everywhere.
* Its view is still **local**: with 3 layers of kernel 9 it sees 25 residues. Two strands that pair in a β-sheet can be far apart in the sequence, and a CNN of this size cannot see that.

The **LSTM** in the next notebook reads the whole protein in order and carries a memory along the chain.

## Exercises

1. Change `N_LAYERS` to 1, 2, 5. How do the receptive field and the Q3 change?
2. Use `KERNEL = 3` with `N_LAYERS = 12`. That gives the same receptive field (25 residues) as the default with many more layers. Is it better or worse?
3. Add `nn.BatchNorm1d(channels)` after each `Conv1d`. Does training get faster?